# Missingness-Aware Trajectory Prediction

Run this notebook in Google Colab after uploading the project folder or placing it in Google Drive. It loads the coordinate trajectory data, evaluates Baseline A constant velocity, trains Baseline B vanilla LSTM encoder-decoder adapted from https://github.com/lkulowski/LSTM_encoder_decoder, trains the proposed missingness-aware LSTM, trains a non-LSTM missingness-aware Transformer, and saves a qualitative plot. The proposed models are residual-over-constant-velocity models: they learn corrections to a constant-velocity forecast rather than predicting the whole future path from scratch. The optimized motion-feature mode adds relative position, velocity, acceleration, observation mask, and time-gap inputs, and the batch table reports both ADE/FDE and navigation action metrics.


In [ ]:
!pip install -q torch numpy matplotlib

## Project Path

Option A: upload this whole project folder to `/content/project`.

Option B: put it in Google Drive, then mount Drive and set `PROJECT_ROOT` to the folder path.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

PROJECT_ROOT = Path('/content/drive/MyDrive/cv_project/project')
DATASET_ROOT = PROJECT_ROOT / 'data' / 'preprocessed' / 'datasets_LMTrajectory'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT exists:', DATASET_ROOT.exists())
if not DATASET_ROOT.exists():
    raise FileNotFoundError('Could not find data/preprocessed/datasets_LMTrajectory. Check PROJECT_ROOT.')

sys.path = [p for p in sys.path if 'cv_project/project' not in p]
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import copy
import json
from argparse import Namespace
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader

from baseline_model import ConstantVelocityPredictor, VanillaLSTMEncoderDecoder
from data.missingness import build_model_inputs, build_motion_model_inputs
from data.trajectory_dataset import TrajectoryDataset
from navigation import decide_navigation_action
from project_model import MissingnessAwareLSTM, MissingnessAwareTransformer
from utils.metrics import metric_dict
from utils.plotting import save_trajectory_plot

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATASET_ROOT = PROJECT_ROOT / 'data' / 'preprocessed' / 'datasets_LMTrajectory'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# To train longer in Colab, change epochs=10 or epochs=20 here.
# Local equivalent: python experiment.py --model missing_lstm --epochs 10 ...
config = Namespace(
    dataset_root=str(DATASET_ROOT),
    split='zara1',
    obs_len=8,
    pred_len=12,
    stride=1,
    batch_size=256,
    hidden_dim=128,
    num_layers=1,
    transformer_layers=2,
    num_heads=4,
    dropout=0.05,
    lr=5e-4,
    loss='huber',
    residual_weight=0.01,
    feature_mode='motion',
    weight_decay=0.0,
    grad_clip=1.0,
    epochs=10,
    baseline_epochs=10,
    proposed_epochs=20,
    missing_mode='random',
    drop_rate=0.3,
    contiguous_len=3,
    teacher_forcing_ratio=0.5,
    teacher_forcing_decay=True,
    device=DEVICE,
    # Runtime-safe default. For the full report, use ['zara1', 'zara2', 'eth', 'hotel', 'univ'].
    splits=['zara1', 'zara2'],
    missingness_conditions=['complete', 'random_0.1', 'random_0.3', 'random_0.5', 'contiguous', 'partial'],
    models=['constant_velocity', 'vanilla_lstm', 'missing_lstm', 'missing_transformer'],
)
print(config)


In [ ]:
def make_loader(phase, shuffle=False):
    dataset = TrajectoryDataset(
        config.dataset_root,
        config.split,
        phase,
        obs_len=config.obs_len,
        pred_len=config.pred_len,
        stride=config.stride,
    )
    return DataLoader(dataset, batch_size=config.batch_size, shuffle=shuffle)

train_loader = make_loader('train', shuffle=True)
val_loader = make_loader('val')
test_loader = make_loader('test')

sample = train_loader.dataset[0]
print('train windows:', len(train_loader.dataset))
print('obs shape:', tuple(sample['obs'].shape))
print('pred shape:', tuple(sample['pred'].shape))

In [ ]:
@torch.no_grad()
def make_inputs(obs, missing_aware=False, missing_mode='complete', drop_rate=0.0):
    if missing_aware and config.feature_mode == 'motion':
        return build_motion_model_inputs(
            obs,
            mode=missing_mode,
            drop_rate=drop_rate,
            contiguous_len=config.contiguous_len,
        )
    features, last_pos, mask = build_model_inputs(
        obs,
        mode=missing_mode,
        drop_rate=drop_rate,
        contiguous_len=config.contiguous_len,
        missing_aware=missing_aware,
    )
    return features, last_pos, mask, features[..., :2]


@torch.no_grad()
def evaluate(model, loader, missing_aware=False, missing_mode='complete', drop_rate=0.0, plot_name=None):
    if isinstance(model, nn.Module):
        model.eval()
    total_ade, total_fde, count = 0.0, 0.0, 0
    correct_actions, true_stop, pred_stop, true_positive_stop = 0, 0, 0, 0
    sample_plot = None
    for batch in loader:
        obs = batch['obs'].to(config.device)
        target = batch['pred'].to(config.device)
        features, last_pos, mask, cv_obs = make_inputs(obs, missing_aware, missing_mode, drop_rate)
        if isinstance(model, ConstantVelocityPredictor):
            pred = model.predict(cv_obs, mask)
        elif missing_aware:
            pred = model(features, last_pos, mask=mask, cv_obs=cv_obs)
        else:
            pred = model(features, last_pos)
        metrics = metric_dict(pred, target)
        batch_size = obs.shape[0]
        total_ade += metrics['ADE'] * batch_size
        total_fde += metrics['FDE'] * batch_size
        count += batch_size
        for pred_row, target_row in zip(pred, target):
            pred_action = decide_navigation_action(pred_row).action
            true_action = decide_navigation_action(target_row).action
            correct_actions += int(pred_action == true_action)
            true_stop += int(true_action == 'stop')
            pred_stop += int(pred_action == 'stop')
            true_positive_stop += int(pred_action == 'stop' and true_action == 'stop')
        if sample_plot is None:
            sample_plot = (obs[0], mask[0], target[0], pred[0])
    result = {
        'ADE': total_ade / count,
        'FDE': total_fde / count,
        'n': count,
        'action_accuracy': correct_actions / count,
        'stop_precision': true_positive_stop / pred_stop if pred_stop else 0.0,
        'stop_recall': true_positive_stop / true_stop if true_stop else 0.0,
    }
    if plot_name and sample_plot:
        plot_path = RESULTS_DIR / plot_name
        save_trajectory_plot(plot_path, *sample_plot, title=json.dumps(result))
        result['plot'] = str(plot_path)
    return result


def train_model(model, missing_aware=False, missing_mode='complete', drop_rate=0.0, epochs=None, checkpoint_name=None):
    model = model.to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    loss_fn = nn.SmoothL1Loss() if config.loss == 'huber' else nn.MSELoss()
    epochs = epochs or config.epochs
    best_val_ade = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        if config.teacher_forcing_decay and epochs > 1:
            teacher_forcing_ratio = config.teacher_forcing_ratio * (1.0 - (epoch - 1) / (epochs - 1))
        else:
            teacher_forcing_ratio = config.teacher_forcing_ratio

        model.train()
        running, count = 0.0, 0
        for batch in train_loader:
            obs = batch['obs'].to(config.device)
            target = batch['pred'].to(config.device)
            features, last_pos, mask, cv_obs = make_inputs(obs, missing_aware, missing_mode, drop_rate)
            if missing_aware:
                pred = model(features, last_pos, target=target, teacher_forcing_ratio=teacher_forcing_ratio, mask=mask, cv_obs=cv_obs)
            else:
                pred = model(features, last_pos, target=target, teacher_forcing_ratio=teacher_forcing_ratio)
            loss = loss_fn(pred, target)
            if missing_aware and config.residual_weight > 0:
                from utils.trajectory_ops import constant_velocity_forecast
                cv_base = constant_velocity_forecast(cv_obs, mask, config.pred_len)
                loss = loss + config.residual_weight * (pred - cv_base).abs().mean()
            optimizer.zero_grad()
            loss.backward()
            if config.grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()
            running += loss.item() * obs.shape[0]
            count += obs.shape[0]

        val = evaluate(model, val_loader, missing_aware, missing_mode, drop_rate)
        if val['ADE'] < best_val_ade:
            best_val_ade = val['ADE']
            best_state = copy.deepcopy(model.state_dict())
            if checkpoint_name:
                torch.save({'model_state': best_state, 'config': vars(config)}, RESULTS_DIR / checkpoint_name)
        print({'epoch': epoch, 'train_loss': running / count, 'teacher_forcing_ratio': teacher_forcing_ratio, 'val_ADE': val['ADE'], 'val_FDE': val['FDE'], 'val_action_accuracy': val['action_accuracy']})

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


In [ ]:
results = []

cv = ConstantVelocityPredictor(pred_len=config.pred_len)
results.append({'model': 'constant_velocity', 'missingness': 'complete', **evaluate(cv, test_loader)})
results.append({'model': 'constant_velocity', 'missingness': 'random_0.3', **evaluate(cv, test_loader, missing_mode='random', drop_rate=0.3)})

baseline_lstm = train_model(
    VanillaLSTMEncoderDecoder(encoder_input_dim=2, output_dim=2, hidden_dim=config.hidden_dim, num_layers=config.num_layers, pred_len=config.pred_len, dropout=config.dropout),
    missing_aware=False,
    missing_mode='complete',
    drop_rate=0.0,
    epochs=config.baseline_epochs,
    checkpoint_name='colab_vanilla_lstm_best.pt',
)
results.append({'model': 'vanilla_lstm', 'missingness': 'complete', **evaluate(baseline_lstm, test_loader)})
results.append({'model': 'vanilla_lstm', 'missingness': 'random_0.3', **evaluate(baseline_lstm, test_loader, missing_mode='random', drop_rate=0.3)})

missing_lstm = train_model(
    MissingnessAwareLSTM(input_dim=8 if config.feature_mode == 'motion' else 4, hidden_dim=config.hidden_dim, num_layers=config.num_layers, pred_len=config.pred_len, dropout=config.dropout),
    missing_aware=True,
    missing_mode=config.missing_mode,
    drop_rate=config.drop_rate,
    epochs=config.proposed_epochs,
    checkpoint_name='colab_missing_lstm_best.pt',
)
results.append({
    'model': 'missing_lstm',
    'missingness': 'random_0.3',
    **evaluate(missing_lstm, test_loader, missing_aware=True, missing_mode='random', drop_rate=0.3, plot_name='colab_missing_lstm_random.png'),
})

missing_transformer = train_model(
    MissingnessAwareTransformer(input_dim=8 if config.feature_mode == 'motion' else 4, hidden_dim=config.hidden_dim, num_layers=config.transformer_layers, num_heads=config.num_heads, pred_len=config.pred_len, dropout=config.dropout),
    missing_aware=True,
    missing_mode=config.missing_mode,
    drop_rate=config.drop_rate,
    epochs=config.proposed_epochs,
    checkpoint_name='colab_missing_transformer_best.pt',
)
results.append({
    'model': 'missing_transformer',
    'missingness': 'random_0.3',
    **evaluate(missing_transformer, test_loader, missing_aware=True, missing_mode='random', drop_rate=0.3, plot_name='colab_missing_transformer_random.png'),
})


In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(results))
except Exception:
    for row in results:
        print(row)

## Robustness Batch Run

This cell runs the broader experiment table across multiple missingness conditions and dataset splits directly in Python. It also saves the best neural checkpoint for each split/model by validation ADE.

Small sanity check settings:

```python
config.splits = ['zara1', 'zara2']
config.models = ['constant_velocity', 'missing_transformer']
config.missingness_conditions = ['random_0.3']
```

Expected: `4` rows and `2` saved checkpoints.

Full report settings:

```python
config.splits = ['zara1', 'zara2', 'eth', 'hotel', 'univ']
config.models = ['constant_velocity', 'vanilla_lstm', 'missing_lstm', 'missing_transformer']
config.missingness_conditions = ['complete', 'random_0.1', 'random_0.3', 'random_0.5', 'contiguous', 'partial']
```

Expected: `120` rows and `15` saved neural checkpoints.


In [ ]:
from argparse import Namespace
from pathlib import Path
import random
import pandas as pd
import torch
from torch import nn

from baseline_model import ConstantVelocityPredictor
from navigation import decide_navigation_action
from data.missingness import build_model_inputs, build_motion_model_inputs
from run_full_experiments import (
    make_loader,
    make_model,
    evaluate_model,
    write_results,
)
from utils.trajectory_ops import constant_velocity_forecast

MIXED_TRAIN_CONDITIONS = [
    {'mode': 'random', 'drop_rate': 0.1, 'contiguous_len': 3},
    {'mode': 'random', 'drop_rate': 0.3, 'contiguous_len': 3},
    {'mode': 'random', 'drop_rate': 0.5, 'contiguous_len': 3},
    {'mode': 'partial', 'drop_rate': 0.3, 'contiguous_len': 3},
]
MISSING_AWARE_VAL_CONDITIONS = ['random_0.3', 'random_0.5', 'partial']


def make_inputs(obs, args, missing_aware, mode, drop_rate, contiguous_len):
    if missing_aware and args.feature_mode == 'motion':
        return build_motion_model_inputs(obs, mode=mode, drop_rate=drop_rate, contiguous_len=contiguous_len)
    features, last_pos, mask = build_model_inputs(obs, mode=mode, drop_rate=drop_rate, contiguous_len=contiguous_len, missing_aware=missing_aware)
    return features, last_pos, mask, features[..., :2]


def validation_score(model, val_loader, args, model_name, split):
    if model_name not in {'missing_lstm', 'missing_transformer'}:
        metrics, _ = evaluate_model(model, val_loader, args, model_name, 'complete', save_plot=False, split=split)
        return metrics['ADE']
    scores = []
    for condition_name in MISSING_AWARE_VAL_CONDITIONS:
        metrics, _ = evaluate_model(model, val_loader, args, model_name, condition_name, save_plot=False, split=split)
        scores.append(metrics['ADE'])
    return sum(scores) / len(scores)


def prediction_loss_with_regularization(pred, target, cv_obs, mask, pred_len, loss_fn, residual_weight):
    loss = loss_fn(pred, target)
    if residual_weight <= 0:
        return loss
    cv_base = constant_velocity_forecast(cv_obs, mask, pred_len)
    residual_penalty = (pred - cv_base).abs().mean()
    return loss + residual_weight * residual_penalty


def train_neural_model_with_checkpoint(args, model, train_loader, val_loader, model_name, split):
    missing_aware = model_name in {'missing_lstm', 'missing_transformer'}
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    loss_fn = nn.SmoothL1Loss() if args.loss == 'huber' else nn.MSELoss()
    best_state = None
    best_val_score = float('inf')
    checkpoint_path = Path(args.output_dir) / f'colab_{split}_{model_name}_best.pt'

    for epoch in range(1, args.epochs + 1):
        if args.teacher_forcing_decay and args.epochs > 1:
            teacher_forcing_ratio = args.teacher_forcing_ratio * (1.0 - (epoch - 1) / (args.epochs - 1))
        else:
            teacher_forcing_ratio = args.teacher_forcing_ratio

        model.train()
        running = 0.0
        count = 0
        for batch in train_loader:
            obs = batch['obs'].to(args.device)
            target = batch['pred'].to(args.device)
            train_params = random.choice(MIXED_TRAIN_CONDITIONS) if missing_aware else {
                'mode': 'complete',
                'drop_rate': 0.0,
                'contiguous_len': 3,
            }
            features, last_pos, mask, cv_obs = make_inputs(
                obs,
                args,
                missing_aware,
                train_params['mode'],
                train_params['drop_rate'],
                train_params['contiguous_len'],
            )
            if missing_aware:
                pred = model(
                    features,
                    last_pos,
                    target=target,
                    teacher_forcing_ratio=teacher_forcing_ratio,
                    mask=mask,
                    cv_obs=cv_obs,
                )
                loss = prediction_loss_with_regularization(
                    pred,
                    target,
                    cv_obs,
                    mask,
                    args.pred_len,
                    loss_fn,
                    args.residual_weight,
                )
            else:
                pred = model(features, last_pos, target=target, teacher_forcing_ratio=teacher_forcing_ratio)
                loss = loss_fn(pred, target)

            optimizer.zero_grad()
            loss.backward()
            if args.grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            running += float(loss.item()) * obs.shape[0]
            count += obs.shape[0]

        val_score = validation_score(model, val_loader, args, model_name, split)
        train_loss = running / max(1, count)
        print({
            'split': split,
            'model': model_name,
            'epoch': epoch,
            'train_loss': train_loss,
            'teacher_forcing_ratio': teacher_forcing_ratio,
            'val_score_ADE': val_score,
        })

        if val_score < best_val_score:
            best_val_score = val_score
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            torch.save(
                {
                    'model_state': best_state,
                    'split': split,
                    'model_name': model_name,
                    'config': vars(config),
                    'best_val_ADE': best_val_score,
                    'validation_conditions': MISSING_AWARE_VAL_CONDITIONS if missing_aware else ['complete'],
                },
                checkpoint_path,
            )
            print(f'Saved best checkpoint: {checkpoint_path}')

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


batch_args = Namespace(
    dataset_root=str(DATASET_ROOT),
    splits=config.splits,
    models=config.models,
    conditions=config.missingness_conditions,
    obs_len=config.obs_len,
    pred_len=config.pred_len,
    stride=config.stride,
    epochs=config.epochs,
    batch_size=config.batch_size,
    hidden_dim=config.hidden_dim,
    num_layers=config.num_layers,
    transformer_layers=config.transformer_layers,
    num_heads=config.num_heads,
    dropout=config.dropout,
    lr=config.lr,
    loss=config.loss,
    residual_weight=config.residual_weight,
    feature_mode=config.feature_mode,
    weight_decay=config.weight_decay,
    grad_clip=config.grad_clip,
    teacher_forcing_ratio=config.teacher_forcing_ratio,
    teacher_forcing_decay=config.teacher_forcing_decay,
    output_dir=str(RESULTS_DIR),
    device=config.device,
    num_workers=0,
    max_plot_rows=8,
)

rows = []
saved_plots = 0

for split in batch_args.splits:
    print(f'\n=== Split: {split} ===')
    train_loader = make_loader(batch_args, split, 'train', shuffle=True)
    val_loader = make_loader(batch_args, split, 'val', shuffle=False)
    test_loader = make_loader(batch_args, split, 'test', shuffle=False)

    for model_name in batch_args.models:
        print(f'\nModel: {model_name}')
        model = make_model(batch_args, model_name)

        if not isinstance(model, ConstantVelocityPredictor):
            model = train_neural_model_with_checkpoint(
                batch_args,
                model,
                train_loader,
                val_loader,
                model_name,
                split,
            )

        for condition_name in batch_args.conditions:
            save_plot = saved_plots < batch_args.max_plot_rows
            result, _ = evaluate_model(
                model,
                test_loader,
                batch_args,
                model_name,
                condition_name,
                save_plot=save_plot,
                split=split,
            )
            if save_plot:
                saved_plots += 1
            rows.append(result)
            print(result)

output_path = RESULTS_DIR / 'full_experiment_results.csv'
write_results(output_path, rows)
print(f'\nSaved {len(rows)} rows to {output_path}')
print(f'Expected rows: {len(batch_args.splits) * len(batch_args.models) * len(batch_args.conditions)}')


In [ ]:
import pandas as pd
from pathlib import Path

full_results_path = RESULTS_DIR / 'full_experiment_results.csv'
full_results = pd.read_csv(full_results_path)

print('rows:', len(full_results))
print('expected rows:', len(config.splits) * len(config.models) * len(config.missingness_conditions))
display(full_results)

summary_cols = ['ADE', 'FDE'] + [col for col in ['action_accuracy', 'stop_precision', 'stop_recall'] if col in full_results.columns]
summary = full_results.groupby(['model', 'missingness'], as_index=False)[summary_cols].mean()
display(summary.sort_values(['missingness', 'ADE']))

checkpoints = sorted(Path(RESULTS_DIR).glob('colab_*_best.pt'))
print('saved colab checkpoints:', len(checkpoints))
for checkpoint in checkpoints:
    print(checkpoint)


## Navigation Demo Plots

The batch run saves representative robot-navigation-style plots under `results/plots/`. Each plot includes observed points, missing observations, future ground truth, predicted future trajectory, the robot, the safety zone, and the selected rule-based action.


In [ ]:
from IPython.display import Image, display

plot_paths = sorted((RESULTS_DIR / 'plots').glob('*.png'))
print('Saved plots:', len(plot_paths))
for plot_path in plot_paths[:4]:
    print(plot_path)
    display(Image(filename=str(plot_path)))
